In [2]:
import sys
from pathlib import Path
import pandas as pd
from contextlib import contextmanager

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

@contextmanager
def full_df_display():
    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.expand_frame_repr", False,
    ):
        yield

%load_ext autoreload
%autoreload 2

path: /workspace


In [3]:
from src.data_download.companies_house import build_download_manifest
company_numbers = ["08948140", "11270200"] # company_name: Finance Advice Centre LTD
FilingList = build_download_manifest(company_numbers)
print(len(FilingList))

13


In [4]:
from src.data_download.companies_house import download_ixbrl_filing
for filing in FilingList:
    download_ixbrl_filing(filing)

2025-10-06, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzQ4Mzg0NDYxN2FkaXF6a2N4.ixbrl
2025-08-26, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzQ3ODU4MjE4N2FkaXF6a2N4.ixbrl
2024-11-27, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzQ0NTExOTQ1NmFkaXF6a2N4.ixbrl
2024-07-01, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzQyNzE3ODU4MGFkaXF6a2N4.ixbrl
2023-10-18, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzM5NzAzMzA5NmFkaXF6a2N4.ixbrl
2023-06-27, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzM4NDI4MDYwOGFkaXF6a2N4.ixbrl
2022-10-21, application/xhtml+xml, /workspace/data/raw/companies_house/11270200/MzM1NjEwMjEzNGFkaXF6a2N4.ixbrl
2022-06-17, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzM0Mjg1MTY1MGFkaXF6a2N4.ixbrl
2021-04-28, application/xhtml+xml, /workspace/data/raw/companies_house/08948140/MzI5OTI3NjMxNGFkaXF6a2N4.ixbrl
2

In [32]:
from src.parsing.ixbrl_loader import load_ixbrl_dataframes

ixbrl_folders = projectRoot / "data" / "raw" / "companies_house"
facts_with_context_dict = load_ixbrl_dataframes(ixbrl_folders)

# Example: display the structure of the facts_with_context for each filing
# for filing_id, facts_with_context in facts_with_context_dict.items():
#     print(f"\nFiling ID: {filing_id}")
#     print("Type:", type(facts_with_context))
#     print("Shape:", facts_with_context.shape)
#     print("Columns:", facts_with_context.columns.tolist())

In [ ]:
# Store to CSV files for inspection
import os

processed_dir = projectRoot / "data" / "processed" / "debug"
os.makedirs(processed_dir, exist_ok=True)

for filing_id, df in facts_with_context_dict.items():
    csv_path = processed_dir / f"{filing_id}.csv"
    df.to_csv(csv_path, index=False)

In [41]:
from src.parsing.facts_extractor import (extract_canonical_facts_dataframes,
                                         save_canonical_facts_to_csv)

extractor = extract_canonical_facts_dataframes(facts_with_context_dict)
print(extractor.keys())

processed_dir = projectRoot / "data" / "processed" / "canonical_facts"
save_canonical_facts_to_csv(extractor, processed_dir)

dict_keys(['MzE4NTE0NDIyM2FkaXF6a2N4', 'MzI1MjkyNzY4N2FkaXF6a2N4', 'MzI4MTcyOTA3NWFkaXF6a2N4', 'MzI5OTI3NjMxNGFkaXF6a2N4', 'MzIyMjY3NDUxN2FkaXF6a2N4', 'MzM0Mjg1MTY1MGFkaXF6a2N4', 'MzM4NDI4MDYwOGFkaXF6a2N4', 'MzQ3ODU4MjE4N2FkaXF6a2N4', 'MzQyNzE3ODU4MGFkaXF6a2N4', 'MzM1NjEwMjEzNGFkaXF6a2N4', 'MzM5NzAzMzA5NmFkaXF6a2N4', 'MzQ0NTExOTQ1NmFkaXF6a2N4', 'MzQ4Mzg0NDYxN2FkaXF6a2N4'])


In [39]:
# get second key without building a list
if not extractor:
    raise ValueError("extractor is empty")
it = iter(extractor)
try:
    next(it)  # skip first
    second_key = next(it)
except StopIteration:
    raise ValueError("extractor has less than 2 items")
print(second_key)

ff = extractor[second_key]["financial_facts"]
narrPol = extractor[second_key]["narrative_policies"]
enCompliance = extractor[second_key]["entity_compliance"]

# sense check entries
ff[ff['name'] == "frs-core:IncreaseFromDepreciationChargeForYearPropertyPlantEquipment"]
ff[ff['name'].str.contains("DepreciationCharge", case=False, na=False)]

MzI1MjkyNzY4N2FkaXF6a2N4


,name,contextRef,unitRef,decimals,value,measure,entity_identifier,entity_scheme,start_date,end_date,instant,scenario,domain
34,frs-core:IncreaseFromDepreciationChargeForYear...,CURRENT_FY_PERIOD_FixturesFittings,GBP,0,97,iso4217:GBP,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None,frs-core
35,frs-core:IncreaseFromDepreciationChargeForYear...,CURRENT_FY_PERIOD_ComputerEquipment,GBP,0,"2,575",iso4217:GBP,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None,frs-core
36,frs-core:IncreaseFromDepreciationChargeForYear...,CURRENT_FY_PERIOD,GBP,0,"2,672",iso4217:GBP,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None,frs-core
